# summary_test (Stage 1 Summary in Colab + Gradio)

This notebook ports **Stage 1** (`run_summary_stage`) from `Dataset Prep/prep_pipeline.ipynb` and adapts it for:

- Colab kernel runtime
- Gradio file upload input (no fixed local scripts path)
- detailed step-by-step progress output
- Gemini connectivity preflight test (before summary)

Workflow recommendation:
1. Run **Test Gemini Connection** first.
2. If PASS, run **Stage 1 Summary**.

**Version:** `v6 fixed-model-paste-input`


In [1]:
# Install dependencies for a clean Colab runtime
!pip -q install -U google-genai gradio pypdf python-docx striprtf tqdm tiktoken


In [2]:
import os
import json
import time
import shutil
import tempfile
import subprocess
from pathlib import Path
from datetime import datetime

from google import genai as google_genai
from pypdf import PdfReader

try:
    from docx import Document as DocxDocument
except Exception:
    DocxDocument = None

try:
    from striprtf.striprtf import rtf_to_text
except Exception:
    rtf_to_text = None

try:
    import tiktoken
except Exception:
    tiktoken = None

import gradio as gr

SUPPORTED_EXTENSIONS = {".pdf", ".doc", ".docx", ".txt", ".rtf"}
FIXED_MODEL_NAME = "gemini-3-flash-preview"


def now_ts() -> str:
    return datetime.now().strftime("%H:%M:%S")


def estimate_tokens(text: str) -> int:
    if not text:
        return 0

    if tiktoken is not None:
        try:
            enc = tiktoken.get_encoding("cl100k_base")
            return len(enc.encode(text))
        except Exception:
            pass

    return max(1, int(len(text) / 4))


def extract_response_text(response) -> str:
    text = getattr(response, "text", None)
    if text and str(text).strip():
        return str(text).strip()

    try:
        candidates = getattr(response, "candidates", None) or []
        if candidates:
            first = candidates[0]
            content = getattr(first, "content", None)
            parts = getattr(content, "parts", None) or []
            chunks = []
            for p in parts:
                t = getattr(p, "text", None)
                if t:
                    chunks.append(str(t))
            if chunks:
                return "\n".join(chunks).strip()
    except Exception:
        pass

    return ""


In [3]:
# Copied and adapted from Dataset Prep/prep_pipeline.ipynb
class UsageTracker:
    """Track Gemini usage metadata across calls."""

    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.total_calls = 0

    def update_and_report(self, usage_metadata, task_name="Task"):
        if not usage_metadata:
            return {
                "task": task_name,
                "input": 0,
                "output": 0,
                "total": 0,
                "grand_total": self.total_input + self.total_output,
            }

        current_input = int(getattr(usage_metadata, "prompt_token_count", 0) or 0)
        current_output = int(getattr(usage_metadata, "candidates_token_count", 0) or 0)
        current_total = current_input + current_output

        self.total_input += current_input
        self.total_output += current_output
        self.total_calls += 1
        grand_total = self.total_input + self.total_output

        return {
            "task": task_name,
            "input": current_input,
            "output": current_output,
            "total": current_total,
            "grand_total": grand_total,
        }

    def summary_text(self) -> str:
        return (
            "\n" + "=" * 40 + "\n"
            + "FINAL USAGE REPORT\n"
            + f"Total API Calls: {self.total_calls}\n"
            + f"Total Input:     {self.total_input}\n"
            + f"Total Output:    {self.total_output}\n"
            + f"Grand Total:     {self.total_input + self.total_output}\n"
            + "=" * 40
        )


class LLMClient:
    """Gemini client wrapper using google-genai SDK."""

    def __init__(self, api_key: str, model_name: str, usage_tracker: UsageTracker):
        self.model_name = model_name
        self.usage_tracker = usage_tracker
        self.client = google_genai.Client(api_key=api_key)

    def generate_content(
        self,
        prompt: str,
        system_instruction: str = None,
        task_tag: str = "Gen",
        request_timeout_sec: int = 180,  # kept for interface compatibility
        max_output_tokens: int = 1800,   # kept for interface compatibility
    ):
        # Keep behavior stable without SDK-specific config dependencies.
        full_prompt = prompt
        if system_instruction and system_instruction.strip():
            full_prompt = f"{system_instruction.strip()}\n\n---\n\n{prompt}"

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=full_prompt,
        )

        usage_row = self.usage_tracker.update_and_report(
            getattr(response, "usage_metadata", None),
            task_name=task_tag,
        )
        text = extract_response_text(response)
        return text, usage_row


class ScriptSummarizerAgent:
    """
    Step 1 summarizer from prep_pipeline, preserving original instruction structure.
    """

    SYSTEM_INSTRUCTION = """
Role: You are a Structural Script Analyst and Narrative Architect.
Task: Compress the provided raw script treatment into a "Uplift-Ready Structural Summary."
Constraints:

Length: Target 1200-1500 tokens.
Preserve: Core conflict, protagonist's internal and external arcs, emotional peaks, and moral/thematic undercurrents.
Remove: Detailed dialogue, minor characters with no plot impact, specific scene descriptions, and fluff.
Specific Focus (The "Mission Keeper" Lens):

Highlight the Protagonist's Agency: How they drive the plot through choices.
Identify Heroine's Journey markers: Internal awakening and reclaiming of power.
Emphasize Moral Dilemmas: Points where the "Humanity Uplifting" principles are tested.
Output Structure (Markdown Headers):

## Logline: A one-sentence summary.
## Thematic Core: The central moral question or "Uplift" theme.
## Character Profile: The protagonist's initial state vs. final state (Arc).
## Structural Breakdown: Key beats (Inciting Incident, Midpoint, Climax, Resolution) with emphasis on emotional transitions.
## Social Context: Any racial, cultural, or gender-specific dynamics relevant to "Dignity & Inclusion."
"""

    def __init__(self, llm_client: LLMClient):
        self.client = llm_client

    def run(self, raw_script: str, filename: str):
        prompt = f"Here is the raw script treatment for '{filename}'. Please summarize it:\n\n{raw_script}"
        return self.client.generate_content(
            prompt,
            system_instruction=self.SYSTEM_INSTRUCTION,
            task_tag=f"Summary|{filename[:10]}",
            request_timeout_sec=180,
            max_output_tokens=1800,
        )


In [4]:
# File extraction functions migrated from prep_pipeline with minimal adaptation

def extract_text_from_pdf(pdf_path: Path) -> str:
    try:
        reader = PdfReader(str(pdf_path))
        text = ""
        for page in reader.pages:
            text += (page.extract_text() or "") + ""
        return text
    except Exception as e:
        raise RuntimeError(f"Error reading PDF {pdf_path}: {e}")


def extract_text_from_docx(docx_path: Path) -> str:
    if DocxDocument is None:
        raise RuntimeError("python-docx is not installed, cannot parse DOCX")

    try:
        doc = DocxDocument(str(docx_path))
        paragraphs = [p.text for p in doc.paragraphs if p.text and p.text.strip()]

        table_texts = []
        for table in doc.tables:
            for row in table.rows:
                cells = [cell.text.strip() for cell in row.cells if cell.text and cell.text.strip()]
                if cells:
                    table_texts.append(" | ".join(cells))

        return "".join(paragraphs + table_texts)
    except Exception as e:
        raise RuntimeError(f"Error reading DOCX {docx_path}: {e}")


def extract_text_from_txt(txt_path: Path) -> str:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]
    for enc in encodings:
        try:
            with open(txt_path, "r", encoding=enc) as f:
                return f.read()
        except Exception:
            continue

    try:
        with open(txt_path, "r", errors="ignore") as f:
            return f.read()
    except Exception as e:
        raise RuntimeError(f"Error reading TXT {txt_path}: {e}")


def _extract_via_textutil(path_obj: Path) -> str:
    if shutil.which("textutil") is None:
        return ""

    try:
        result = subprocess.run(
            ["textutil", "-convert", "txt", "-stdout", str(path_obj)],
            capture_output=True,
            text=True,
            check=False,
        )
        if result.returncode == 0:
            return result.stdout or ""
    except Exception:
        pass

    return ""


def _extract_via_soffice(path_obj: Path) -> str:
    soffice_cmd = shutil.which("soffice") or shutil.which("libreoffice")
    if soffice_cmd is None:
        return ""

    try:
        with tempfile.TemporaryDirectory() as tmpdir:
            result = subprocess.run(
                [
                    soffice_cmd,
                    "--headless",
                    "--convert-to",
                    "txt:Text",
                    "--outdir",
                    tmpdir,
                    str(path_obj),
                ],
                capture_output=True,
                text=True,
                check=False,
            )
            if result.returncode != 0:
                return ""

            txt_path = Path(tmpdir) / f"{path_obj.stem}.txt"
            if txt_path.exists():
                return extract_text_from_txt(txt_path)
    except Exception:
        pass

    return ""


def extract_text_from_doc(doc_path: Path) -> str:
    text = _extract_via_textutil(doc_path)
    if text.strip():
        return text

    text = _extract_via_soffice(doc_path)
    if text.strip():
        return text

    raise RuntimeError(
        f"Unable to parse DOC file: {doc_path}. Install textutil (macOS) or LibreOffice."
    )


def extract_text_from_rtf(rtf_path: Path) -> str:
    if rtf_to_text is not None:
        try:
            raw = extract_text_from_txt(rtf_path)
            return rtf_to_text(raw)
        except Exception as e:
            raise RuntimeError(f"Error reading RTF {rtf_path}: {e}")

    text = _extract_via_textutil(rtf_path)
    if text.strip():
        return text

    raise RuntimeError(
        f"Unable to parse RTF file: {rtf_path}. Install striprtf or enable textutil."
    )


def extract_text_from_file(file_path: Path) -> str:
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        return extract_text_from_pdf(file_path)
    if suffix == ".docx":
        return extract_text_from_docx(file_path)
    if suffix == ".doc":
        return extract_text_from_doc(file_path)
    if suffix == ".txt":
        return extract_text_from_txt(file_path)
    if suffix == ".rtf":
        return extract_text_from_rtf(file_path)

    raise RuntimeError(f"Unsupported extension: {suffix}")


def normalize_uploaded_files(uploaded_files):
    normalized = []
    for item in uploaded_files or []:
        if isinstance(item, str):
            p = Path(item)
        elif hasattr(item, "name"):
            p = Path(item.name)
        elif isinstance(item, dict) and "name" in item:
            p = Path(item["name"])
        else:
            continue

        if p.exists() and p.is_file():
            normalized.append(p)

    unique = []
    seen = set()
    for p in normalized:
        k = str(p.resolve())
        if k not in seen:
            seen.add(k)
            unique.append(p)
    return unique


In [5]:
# Gemini connectivity test + in-memory summary stage (google-genai only, no JSON)

def run_gemini_connectivity_test(api_key: str, timeout_sec: int = 30):
    logs = []
    sample_text = ""
    ok = False
    model_name = FIXED_MODEL_NAME

    def add(msg: str):
        logs.append(f"[{now_ts()}] {msg}")

    api_key = (api_key or "").strip()
    if not api_key:
        add("Missing GOOGLE_API_KEY")
        return False, "\n".join(logs), ""

    try:
        add(f"google-genai test start: model={model_name}")
        client = google_genai.Client(api_key=api_key)
        response = client.models.generate_content(
            model=model_name,
            contents="Reply with exactly GEMINI_OK",
        )

        text = extract_response_text(response)
        if text:
            ok = True
            sample_text = text[:500]
            add(f"google-genai success. reply={sample_text!r}")
        else:
            add("google-genai returned empty response.")
    except Exception as e:
        add(f"google-genai failed: {type(e).__name__}: {e}")

    return ok, "\n".join(logs), sample_text


def run_connectivity_test_ui(api_key: str):
    ok, logs_text, sample_text = run_gemini_connectivity_test(api_key, timeout_sec=30)
    status = "PASS: Gemini reachable" if ok else "FAIL: Gemini not reachable"
    return status, logs_text, sample_text


def run_summary_stage(
    api_key: str,
    uploaded_files,
    pasted_text: str,
):
    progress_text = "0%"
    logs = []
    latest_summary_md = "_No summary generated yet._"
    usage_report = ""
    model_name = FIXED_MODEL_NAME

    def snapshot():
        return (
            progress_text,
            "\n".join(logs),
            latest_summary_md,
            usage_report,
        )

    def log(step: str, message: str, pct=None):
        nonlocal progress_text
        if pct is not None:
            progress_text = f"{pct:.1f}%"
        logs.append(f"[{now_ts()}] {step} | {message}")

    log("STEP 0.1", "Pipeline started.", 0.0)
    yield snapshot()

    api_key = (api_key or "").strip()
    if not api_key:
        log("STEP 0.2", "Missing GOOGLE_API_KEY. Provide key in UI and retry.", 0.0)
        yield snapshot()
        return

    files = normalize_uploaded_files(uploaded_files)
    pasted_text = (pasted_text or "").strip()

    log("STEP 0.3", f"Received uploaded entries: {len(uploaded_files or [])}; usable files: {len(files)}", 2.0)
    yield snapshot()

    if pasted_text:
        log("STEP 0.4", f"Received pasted text. chars={len(pasted_text)}, tokens_est={estimate_tokens(pasted_text)}", 3.0)
        yield snapshot()

    if not files and not pasted_text:
        log("STEP 0.5", "No valid uploaded file or pasted text found.", 2.0)
        yield snapshot()
        return

    log("STEP 1.1", f"Using google-genai client. Fixed model: {model_name}", 5.0)
    yield snapshot()

    log("STEP 1.2", "Running Gemini connectivity preflight test.", 6.0)
    yield snapshot()

    ok, preflight_logs, sample_text = run_gemini_connectivity_test(api_key, timeout_sec=30)
    for line in preflight_logs.splitlines():
        logs.append(line)
    if sample_text:
        logs.append(f"[{now_ts()}] STEP 1.3 | Preflight sample reply: {sample_text[:200]}")

    if not ok:
        log("STEP 1.4", "Preflight failed. Stop before summary stage.", 6.0)
        yield snapshot()
        return

    log("STEP 1.5", "Preflight passed. Continue to summary stage.", 8.0)
    yield snapshot()

    tracker = UsageTracker()
    client_flash = LLMClient(api_key=api_key, model_name=model_name, usage_tracker=tracker)
    summarizer = ScriptSummarizerAgent(client_flash)
    log("STEP 2.1", "Initialized UsageTracker, LLMClient, ScriptSummarizerAgent.", 10.0)
    yield snapshot()

    work_items = []
    for script_path in files:
        work_items.append({
            "source_type": "file",
            "name": script_path.name,
            "path": script_path,
        })

    if pasted_text:
        work_items.append({
            "source_type": "pasted_text",
            "name": "Pasted Text Input",
            "text": pasted_text,
        })

    total_items = len(work_items)
    completed = 0
    skipped = 0
    failed = 0
    summary_blocks = []

    for idx, item in enumerate(work_items, start=1):
        base_pct = 10.0 + ((idx - 1) / max(total_items, 1)) * 84.0
        source_type = item["source_type"]
        display_name = item["name"]

        log(f"STEP 3.{idx}.1", f"Start item {idx}/{total_items}: {display_name} ({source_type})", base_pct)
        yield snapshot()

        raw_text = ""

        if source_type == "file":
            script_path = item["path"]
            suffix = script_path.suffix.lower()
            if suffix not in SUPPORTED_EXTENSIONS:
                skipped += 1
                log(f"STEP 3.{idx}.2", f"Unsupported extension {suffix}; skipped.", base_pct + 1.0)
                yield snapshot()
                continue

            try:
                log(f"STEP 3.{idx}.3", "Extracting raw text from uploaded file.", base_pct + 2.0)
                yield snapshot()

                raw_text = extract_text_from_file(script_path)
                raw_chars = len(raw_text or "")
                raw_tokens_est = estimate_tokens(raw_text or "")

                log(
                    f"STEP 3.{idx}.4",
                    f"Extraction done. chars={raw_chars}, estimated_tokens={raw_tokens_est}",
                    base_pct + 6.0,
                )
                yield snapshot()

            except Exception as e:
                failed += 1
                log(f"STEP 3.{idx}.E1", f"Text extraction failed: {e}", base_pct + 8.0)
                yield snapshot()
                continue

        else:
            raw_text = item["text"]
            raw_chars = len(raw_text)
            raw_tokens_est = estimate_tokens(raw_text)
            log(
                f"STEP 3.{idx}.3",
                f"Using pasted text. chars={raw_chars}, estimated_tokens={raw_tokens_est}",
                base_pct + 6.0,
            )
            yield snapshot()

        if not raw_text or not raw_text.strip():
            skipped += 1
            log(f"STEP 3.{idx}.5", "Input text is empty; skipped.", base_pct + 8.0)
            yield snapshot()
            continue

        try:
            log(f"STEP 3.{idx}.6", "Sending summary request to Gemini.", base_pct + 10.0)
            yield snapshot()

            start_t = time.time()
            summary_text, usage_row = summarizer.run(raw_text, display_name)
            elapsed = time.time() - start_t

            if not summary_text or not summary_text.strip():
                failed += 1
                log(f"STEP 3.{idx}.E2", "Gemini returned empty summary.", base_pct + 20.0)
                yield snapshot()
                continue

            summary_chars = len(summary_text)
            summary_tokens_est = estimate_tokens(summary_text)

            usage_msg = (
                f"usage(input={usage_row['input']}, output={usage_row['output']}, "
                f"total={usage_row['total']}, cumulative={usage_row['grand_total']})"
            )
            log(
                f"STEP 3.{idx}.7",
                f"Gemini summary done in {elapsed:.2f}s. summary_chars={summary_chars}, "
                f"summary_tokens_est={summary_tokens_est}, {usage_msg}",
                base_pct + 24.0,
            )
            yield snapshot()

        except Exception as e:
            failed += 1
            log(f"STEP 3.{idx}.E3", f"Gemini summary request failed: {type(e).__name__}: {e}", base_pct + 20.0)
            yield snapshot()
            continue

        summary_blocks.append(
            "\n\n---\n\n".join([
                f"### Source: `{display_name}` ({source_type})",
                f"- Input tokens (est): `{raw_tokens_est}`",
                f"- Summary tokens (est): `{summary_tokens_est}`",
                "#### Gemini Summary",
                summary_text,
            ])
        )
        latest_summary_md = "\n\n".join(summary_blocks)

        completed += 1
        log(
            f"STEP 3.{idx}.8",
            f"Item complete. completed={completed}, skipped={skipped}, failed={failed}",
            min(95.0, base_pct + 30.0),
        )
        yield snapshot()

    usage_report = tracker.summary_text()

    log("STEP 4.1", f"Summary stage complete. completed={completed}, skipped={skipped}, failed={failed}", 99.0)
    yield snapshot()

    log("STEP 4.2", "Done. Output is shown in the Summary Content panel only.", 100.0)
    yield snapshot()


In [6]:
# Gradio interface for Gemini test + Stage 1 summary (google-genai only)
with gr.Blocks(title="summary_test: Stage 1 Summary", css="""
#run-btn {font-weight: 700;}
#test-btn {font-weight: 700;}
""") as demo:
    gr.Markdown("""
    ## Stage 1 Summary Test (Upload/Paste -> Gemini -> Structured Summary)
    Use **Test Gemini Connection** first. If PASS, run the summary stage.

    - Model is fixed: `gemini-3-flash-preview`
    - No JSON output: summaries are displayed only in the Summary panel.
    """)

    with gr.Row():
        api_key_in = gr.Textbox(
            label="GOOGLE_API_KEY",
            type="password",
            placeholder="Paste your Gemini API key",
        )
        fixed_model_out = gr.Textbox(
            label="Gemini Model (Fixed)",
            value=FIXED_MODEL_NAME,
            interactive=False,
        )

    test_btn = gr.Button("Test Gemini Connection", elem_id="test-btn", variant="secondary")
    test_status_out = gr.Textbox(label="Gemini Test Status")
    test_logs_out = gr.Textbox(label="Gemini Test Logs", lines=10, max_lines=16)
    test_reply_out = gr.Textbox(label="Gemini Test Reply Preview", lines=4)

    with gr.Row():
        files_in = gr.Files(
            label="Upload scripts (pdf/doc/docx/txt/rtf)",
            file_types=[".pdf", ".doc", ".docx", ".txt", ".rtf"],
            type="filepath",
        )

    pasted_text_in = gr.Textbox(
        label="Paste Script Text (New)",
        placeholder="Paste script/treatment text here. It will be summarized by Gemini.",
        lines=14,
        max_lines=24,
    )

    run_btn = gr.Button("Run Stage 1 Summary", elem_id="run-btn", variant="primary")

    with gr.Row():
        progress_out = gr.Textbox(label="Progress", value="0%")
        usage_out = gr.Textbox(label="Usage Summary", lines=8)

    logs_out = gr.Textbox(label="Detailed Step Logs", lines=24, max_lines=40)
    latest_summary_out = gr.Markdown(label="Summary Content (Gemini output)")

    test_btn.click(
        fn=run_connectivity_test_ui,
        inputs=[api_key_in],
        outputs=[test_status_out, test_logs_out, test_reply_out],
        show_progress=True,
    )

    run_btn.click(
        fn=run_summary_stage,
        inputs=[api_key_in, files_in, pasted_text_in],
        outputs=[
            progress_out,
            logs_out,
            latest_summary_out,
            usage_out,
        ],
        show_progress=True,
    )


demo.launch(share=True)


/tmp/ipython-input-46110/820564227.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="summary_test: Stage 1 Summary", css="""


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://da338641b04f4cf8b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
